### Supplement 4: Retrieval (`4-retrieval.py`)

The same tool loop as Supplement 3, but the tool reads your own data (`kb.json`, a small FAQ for an online store) instead of calling a weather API.

**The key point:** The model has never seen your store's policies. It can answer only because your code puts the knowledge-base text into `messages` as a tool result before call 2. Fetching the right text and putting it in front of the model is the core idea of retrieval (RAG, retrieval-augmented generation).

```
Call 1     create(messages, tools)
           └─> a request, not an answer: run search_kb(question)
Your code  search_kb(...) returns everything in kb.json
           └─> append the request and the result to messages
Call 2     parse(messages, tools, response_format=KBResponse)
           └─> the answer, plus source = the id of the record it used
Call 3     a new question that kb.json can't answer
           └─> does the model still ask for the tool?
```

OpenAI docs: [function calling](https://platform.openai.com/docs/guides/function-calling).

#### 1. Setup
`4-retrieval.py` never calls `load_dotenv()`, so it only works when `OPENAI_API_KEY` is already set in your system environment. The notebook adds `load_dotenv()` so the key comes from `.env`, as in Supplements 1–3.

In [1]:
import json
import os
from openai import OpenAI
from pydantic import BaseModel, Field

from dotenv import load_dotenv  # added: 4-retrieval.py doesn't load .env

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

#### 2. The knowledge base and `search_kb`
`kb.json` holds three question/answer records, each with an `id`. `search_kb` is a mock: it ignores `question` and returns the whole file, so picking the right record is left to the model. A real retrieval system would search and return only the matching records.

In [2]:
def search_kb(question: str):
    """
    Load the whole knowledge base from the JSON file.
    (This is a mock function for demonstration purposes, we don't search)
    """
    with open("kb.json", "r") as f:
        return json.load(f)

In [3]:
kb = search_kb("any question")  # the argument is ignored

print(type(kb))
print(json.dumps(kb, indent=2))

<class 'dict'>
{
  "records": [
    {
      "id": 1,
      "question": "What is the return policy?",
      "answer": "Items can be returned within 30 days of purchase with original receipt. Refunds will be processed to the original payment method within 5-7 business days."
    },
    {
      "id": 2,
      "question": "Do you ship internationally?",
      "answer": "Yes, we ship to over 50 countries worldwide. International shipping typically takes 7-14 business days and costs vary by destination. Please note that customs fees may apply."
    },
    {
      "id": 3,
      "question": "What payment methods do you accept?",
      "answer": "We accept Visa, Mastercard, American Express, PayPal, and Apple Pay. All payments are processed securely through our encrypted payment system."
    }
  ]
}


#### 3. The tool definition
Built like `get_weather`'s in Supplement 3. Its one parameter is `question`, a string the model fills in.

In [4]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_kb",
            "description": "Get the answer to the user's question from the knowledge base.",
            "parameters": {
                "type": "object",
                "properties": {
                    "question": {"type": "string"},
                },
                "required": ["question"],
                "additionalProperties": False,
            },
            "strict": True,
        },
    }
]

#### 4. Call 1: the model asks for `search_kb`
The request doesn't contain the return policy. As in Supplement 3, the reply is a tool request: `finish_reason` is `'tool_calls'`, `content` is `None`, and `arguments` is a JSON string holding the question the model wants to look up.

In [5]:
messages = [
    {"role": "system", "content": "You are a helpful assistant that answers questions from the knowledge base about our e-commerce store."},
    {"role": "user", "content": "What is the return policy?"},
]

completion = client.chat.completions.create(
    model="gpt-5-nano",
    messages=messages,
    tools=tools,
)

In [6]:
print(type(completion))
print(completion.model_dump_json(indent=2))

<class 'openai.types.chat.chat_completion.ChatCompletion'>
{
  "id": "chatcmpl-EPu5BFXMih89hazlNr8jfKkh6TaGg",
  "choices": [
    {
      "finish_reason": "tool_calls",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": null,
        "refusal": null,
        "role": "assistant",
        "annotations": [],
        "audio": null,
        "function_call": null,
        "tool_calls": [
          {
            "id": "call_wZIWh2ssIIughsMOAx6vn6wX",
            "function": {
              "arguments": "{\"question\":\"What is the return policy?\"}",
              "name": "search_kb"
            },
            "type": "function"
          }
        ]
      }
    }
  ],
  "created": 1789842489,
  "model": "gpt-5-nano-2025-08-07",
  "object": "chat.completion",
  "metadata": null,
  "moderation": null,
  "service_tier": "default",
  "system_fingerprint": null,
  "usage": {
    "completion_tokens": 93,
    "prompt_tokens": 158,
    "total_tokens": 251,
    "completio

In [7]:
message = completion.choices[0].message

print("finish_reason:", completion.choices[0].finish_reason)
print("content:      ", message.content)
for tool_call in message.tool_calls:
    print("tool call:")
    print("   id:        ", tool_call.id)
    print("   name:      ", tool_call.function.name)
    print("   arguments: ", repr(tool_call.function.arguments), f"({type(tool_call.function.arguments).__name__})")

finish_reason: tool_calls
content:       None
tool call:
   id:         call_wZIWh2ssIIughsMOAx6vn6wX
   name:       search_kb
   arguments:  '{"question":"What is the return policy?"}' (str)


#### 5. Your code runs `search_kb`
The same loop as Supplement 3. `result` is all of `kb.json`, and `json.dumps(result)` turns it into one string for the `"tool"` message. That string is the only place the store's policies appear in call 2.

In [8]:
def call_function(name, args):
    if name == "search_kb":
        return search_kb(**args)


for tool_call in completion.choices[0].message.tool_calls:
    name = tool_call.function.name
    args = json.loads(tool_call.function.arguments)
    messages.append(completion.choices[0].message)

    result = call_function(name, args)
    messages.append(
        {"role": "tool", "tool_call_id": tool_call.id, "content": json.dumps(result)}
    )

In [9]:
print("name:  ", name, f"({type(name).__name__})")
print("args:  ", args, f"({type(args).__name__})")
print("result:", len(result["records"]), "records, the whole of kb.json", f"({type(result).__name__})")

name:   search_kb (str)
args:   {'question': 'What is the return policy?'} (dict)
result: 3 records, the whole of kb.json (dict)


In [10]:
for i, m in enumerate(messages, start=1):
    print(f"--- entry {i}: {type(m).__name__}")
    print(m)

--- entry 1: dict
{'role': 'system', 'content': 'You are a helpful assistant that answers questions from the knowledge base about our e-commerce store.'}
--- entry 2: dict
{'role': 'user', 'content': 'What is the return policy?'}
--- entry 3: ChatCompletionMessage
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_wZIWh2ssIIughsMOAx6vn6wX', function=Function(arguments='{"question":"What is the return policy?"}', name='search_kb'), type='function')])
--- entry 4: dict
{'role': 'tool', 'tool_call_id': 'call_wZIWh2ssIIughsMOAx6vn6wX', 'content': '{"records": [{"id": 1, "question": "What is the return policy?", "answer": "Items can be returned within 30 days of purchase with original receipt. Refunds will be processed to the original payment method within 5-7 business days."}, {"id": 2, "question": "Do you ship internationally?", "answer": "Yes, we ship to over 50 cou

#### 6. Call 2: the answer and its source
`KBResponse` asks for two fields: `answer`, and `source`, the `id` of the record the answer came from. Because `source` is a record `id`, you can look that record up in `kb.json` and compare it with the answer; the last cell of this step does that.

In [11]:
class KBResponse(BaseModel):
    answer: str = Field(description="The answer to the user's question.")
    source: int = Field(description="The record id of the answer.")

In [12]:
completion_2 = client.chat.completions.parse(
    model="gpt-5-nano",
    messages=messages,
    tools=tools,
    response_format=KBResponse,
)

In [13]:
print(type(completion_2))
# warnings=False hides a harmless Pydantic warning about the `parsed` field; it is still printed.
print(completion_2.model_dump_json(indent=2, warnings=False))

<class 'openai.types.chat.parsed_chat_completion.ParsedChatCompletion[TypeVar]'>
{
  "id": "chatcmpl-EPu5DFgapVexJPgMOxNlfPicpCi7J",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "{\"answer\":\"Items can be returned within 30 days of purchase with the original receipt. Refunds will be processed to the original payment method within 5-7 business days.\",\"source\":1}",
        "refusal": null,
        "role": "assistant",
        "annotations": [],
        "audio": null,
        "function_call": null,
        "tool_calls": null,
        "parsed": {
          "answer": "Items can be returned within 30 days of purchase with the original receipt. Refunds will be processed to the original payment method within 5-7 business days.",
          "source": 1
        }
      }
    }
  ],
  "created": 1789842491,
  "model": "gpt-5-nano-2025-08-07",
  "object": "chat.completion",
  "metadata": null,
  "moderation":

In [14]:
final_response = completion_2.choices[0].message.parsed

print("type:  ", type(final_response).__name__)
print("answer:", final_response.answer)
print("source:", final_response.source)
print()
for record in kb["records"]:
    if record["id"] == final_response.source:
        print(f"kb.json record {record['id']}:", record["answer"])

type:   KBResponse
answer: Items can be returned within 30 days of purchase with the original receipt. Refunds will be processed to the original payment method within 5-7 business days.
source: 1

kb.json record 1: Items can be returned within 30 days of purchase with original receipt. Refunds will be processed to the original payment method within 5-7 business days.


#### 7. Call 3: a question the knowledge base can't answer
A new conversation asks about the weather in Tokyo, with the same `tools` and no `response_format`. The comment in `4-retrieval.py` calls this a "Question that doesn't trigger the tool", but calling a tool is the model's choice, not a rule. Re-running this cell can give either result; two runs on 2026-09-19 gave one of each. Check `finish_reason`:
- `'stop'`: the model answered directly, in `content`.
- `'tool_calls'`: it asked for `search_kb`, `content` is `None`, and this code never runs the tool, so there is no answer. The tool call also has `parsed_arguments` (the arguments already turned into a dict), because this call uses `parse()`.

In [15]:
messages = [
    {"role": "system", "content": "You are a helpful assistant that answers questions from the knowledge base about our e-commerce store."},
    {"role": "user", "content": "What is the weather in Tokyo?"},
]

completion_3 = client.chat.completions.parse(
    model="gpt-5-nano",
    messages=messages,
    tools=tools,
)

In [16]:
print(type(completion_3))
# warnings=False hides a harmless Pydantic warning about the `parsed` field; it is still printed.
print(completion_3.model_dump_json(indent=2, warnings=False))

<class 'openai.types.chat.parsed_chat_completion.ParsedChatCompletion[TypeVar]'>
{
  "id": "chatcmpl-EPu5GilA0nAyIMK6TPu5ecQvD39Sv",
  "choices": [
    {
      "finish_reason": "tool_calls",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": null,
        "refusal": null,
        "role": "assistant",
        "annotations": [],
        "audio": null,
        "function_call": null,
        "tool_calls": [
          {
            "id": "call_5H9TQXCBjwWtURBQNX2ZLiTB",
            "function": {
              "arguments": "{\"question\":\"What is the weather in Tokyo?\"}",
              "name": "search_kb",
              "parsed_arguments": {
                "question": "What is the weather in Tokyo?"
              }
            },
            "type": "function"
          }
        ],
        "parsed": null
      }
    }
  ],
  "created": 1789842494,
  "model": "gpt-5-nano-2025-08-07",
  "object": "chat.completion",
  "metadata": null,
  "moderation": null,
  "s

In [17]:
print("finish_reason:", completion_3.choices[0].finish_reason)
print("content:      ", completion_3.choices[0].message.content)
print("tool_calls:   ", completion_3.choices[0].message.tool_calls)

finish_reason: tool_calls
content:       None
tool_calls:    [ParsedFunctionToolCall(id='call_5H9TQXCBjwWtURBQNX2ZLiTB', function=ParsedFunction(arguments='{"question":"What is the weather in Tokyo?"}', name='search_kb', parsed_arguments={'question': 'What is the weather in Tokyo?'}), type='function')]
